# 03. Validar labels Splink

O modelo do [`02_deduplicar_splink.ipynb`](02_deduplicar_splink.ipynb) é treinado
sem ver a coorte. Aqui a coorte entra só como amostra de **verdade conhecida**
(qualidade do scoring no corte 0,95) — não como funil operacional. O funil
Censo → CPF ouro está em [`04_validar_lista_ouro.ipynb`](04_validar_lista_ouro.ipynb).

**Vocabulário (decisão, não verdade):**

- **positivo** = `match_probability ≥ 0,95` (o sistema aceitou o par);
- **negativo** = `match_probability < 0,95` (o sistema recusou);
- um positivo pode ser verdadeiro ou **falso positivo**.

A labels table do Splink (coluna obrigatória `clerical_match_score`) mistura duas
fontes independentes do score:

- `1.0` — **match conhecido**: par ouro 1:1 com os dois lados no subset;
- `0.0` — **par distinto conhecido**: Censo de A × CPF de B (pessoas distintas
  na ouro) que colidem em ≥1 blocking rule. Sem score ainda; se o modelo der
  ≥ 0,95, esse par vira **positivo** (FP na amostra).

A ouro é amostra, não o universo de todos os matches. Precision populacional
não se mede aqui.

**Pré-requisito:** NB02 (`splink_model.json`, `splink_predictions.parquet`).


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from IPython.display import display

from config import (
    COHORT_DEDUP_ARQUIVO,
    METRICAS_COHORT,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    TABELA_LIMPA,
    drop_splink_temp_tables,
    get_connection,
    get_splink_db_api,
    materialize_gt_no_subset,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

print_paths()
require_input(COHORT_DEDUP_ARQUIVO, label='COHORT')
require_input(SPLINK_MODEL_JSON, label='SPLINK_MODEL_JSON (rode o NB02 antes)')

con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)


## 1. Matches conhecidos (ouro 1:1 no subset)

A coorte diz: este `PERSON_ID_CENSO` é a mesma pessoa que este `CPF_NORM`.
Só pares **1:1** (um Censo ↔ um CPF) e com **os dois lados** em `registro_limpo`
viram `clerical_match_score = 1.0`. Diagnósticos do universo ouro ficam no NB04.


In [ ]:
counts = materialize_gt_no_subset(con, cohort_parquet=COHORT_DEDUP_ARQUIVO)
n_gt = counts['n_gt_no_subset']
print('Matches conhecidos no subset (rótulo 1):', f'{n_gt:,}')
if n_gt == 0:
    raise RuntimeError(
        'Nenhum par da coorte caiu no subset — confira o filtro geográfico do NB00.'
    )


## 2. Labels table: matches conhecidos + pares distintos conhecidos

`clerical_match_score` é o nome que o Splink exige. **Não** é revisão clerical
e **não** é a classe positivo/negativo do corte 0,95.

Pares distintos: atributos do **Censo de A** vs **CPF de B** (A ≠ B na ouro),
nas mesmas 5 regras OR de blocking do NB02. Quem não passa em nenhuma regra
nunca é pontuado — incluir esses pares na labels table infla TN artificiais.

Até 5 pares distintos por âncora (`N_PARES_DISTINTOS_POR_ANCORA`).


In [ ]:
N_PARES_DISTINTOS_POR_ANCORA = 5

# Pares distintos conhecidos: colisão Censo(A) × CPF(B) nas blocking rules
# (não Censo×Censo). Em link_only a labels table exige source_dataset_l/r.
con.execute(f'''
CREATE OR REPLACE TABLE pares_distintos AS
SELECT
    source_dataset_l, unique_id_l, source_dataset_r, unique_id_r
FROM (
    SELECT
        'censo' AS source_dataset_l,
        ga.unique_id_censo AS unique_id_l,
        'cpf' AS source_dataset_r,
        gb.unique_id_cpf AS unique_id_r,
        row_number() OVER (PARTITION BY ga.unique_id_censo ORDER BY random()) AS rn
    FROM gt_no_subset ga
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = ga.unique_id_censo
    JOIN gt_no_subset gb ON gb.person_id_censo <> ga.person_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = gb.unique_id_cpf
    WHERE (
        (ca.primeiro_nome_phon = pb.primeiro_nome_phon
         AND ca.ultimo_nome_phon = pb.ultimo_nome_phon
         AND ca.sexo = pb.sexo)
        OR (ca.ultimo_nome_phon = pb.ultimo_nome_phon
            AND ca.data_nascimento = pb.data_nascimento)
        OR (ca.primeiro_nome_phon = pb.primeiro_nome_phon
            AND ca.data_nascimento = pb.data_nascimento)
        OR (ca.uf = pb.uf
            AND ca.ultimo_nome_phon = pb.ultimo_nome_phon
            AND ca.sexo = pb.sexo)
        OR (ca.uf = pb.uf
            AND ca.primeiro_nome_phon = pb.primeiro_nome_phon
            AND ca.sexo = pb.sexo)
    )
)
WHERE rn <= {N_PARES_DISTINTOS_POR_ANCORA}
''')

con.execute('''
CREATE OR REPLACE TABLE splink_labels AS
SELECT
    'censo' AS source_dataset_l,
    unique_id_censo AS unique_id_l,
    'cpf' AS source_dataset_r,
    unique_id_cpf AS unique_id_r,
    1.0 AS clerical_match_score
FROM gt_no_subset
UNION
SELECT
    source_dataset_l, unique_id_l, source_dataset_r, unique_id_r,
    0.0 AS clerical_match_score
FROM pares_distintos
''')

display(con.execute('''
SELECT
    CASE clerical_match_score
        WHEN 1.0 THEN 'match conhecido (ouro)'
        WHEN 0.0 THEN 'par distinto conhecido'
    END AS tipo,
    COUNT(*) AS n
FROM splink_labels GROUP BY 1 ORDER BY tipo
''').df())

n_distintos = con.execute('SELECT COUNT(*) FROM pares_distintos').fetchone()[0]
if n_distintos == 0:
    print(
        'AVISO: nenhum par distinto conhecido gerado. Com poucos pares no subset '
        'não há base para FP na amostra — amplie o recorte geográfico.'
    )
df_labels = con.execute('SELECT * FROM splink_labels').df()


## 2b. Pares distintos que caíram em positivos (≥ 0,95)

O parquet do NB02 tem pares com score ≥ 0,5 (`predict`). Esta célula **relê**
com `WHERE match_probability >= 0,95` e faz JOIN com `pares_distintos`.

O número é: quantos pares distintos conhecidos **existem nessa tabela filtrada**.
Não reavalia blocking (já foi o critério de entrada). Pares com 0,5 ≤ score < 0,95
estão no parquet do NB02 mas **não** entram no numerador.


In [ ]:
from config import SPLINK_PREDICTIONS

require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o NB02 antes)')

THRESHOLD_AVALIACAO = 0.95

con.execute(f'''
CREATE OR REPLACE TABLE splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_l,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_r,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
WHERE match_probability >= {THRESHOLD_AVALIACAO}
''')

cov = con.execute('''
SELECT
    (SELECT COUNT(*) FROM pares_distintos) AS n_pares_distintos,
    COUNT(*) AS n_distintos_em_positivos_095,
    ROUND(
        100.0 * COUNT(*) / NULLIF((SELECT COUNT(*) FROM pares_distintos), 0),
        2
    ) AS pct_distintos_em_positivos_095
FROM pares_distintos d
JOIN splink_predictions p
  ON p.unique_id_l = d.unique_id_l AND p.unique_id_r = d.unique_id_r
''').df()
display(cov)


## 3. Carregar o modelo treinado

O `Linker` é reconstruído a partir do JSON salvo pelo NB02 — nenhum parâmetro é
reestimado aqui, então a coorte não influencia o modelo.


In [ ]:
import json

with open(SPLINK_MODEL_JSON, 'r') as file:
    data = json.load(file)

data['retain_intermediate_calculation_columns'] = True
data


In [ ]:
from splink import Linker

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')

linker = Linker(
    ['splink_censo', 'splink_cpf'],
    data,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)
labels_sdf = linker.table_management.register_labels_table(df_labels, overwrite=True)
print(f'Labels registradas: {len(df_labels):,}')


## 4. Accuracy por threshold

Curva de precision, recall e F1 sobre a labels table (amostra rotulada, não o
universo). O corte operacional usado abaixo é 0,95.


In [ ]:
linker.evaluation.accuracy_analysis_from_labels_table(
    labels_sdf,
    output_type='accuracy',
    match_weight_round_to_nearest=0.02,
)


## 5. Erros de predição no corte 0,95

Waterfall dos pares em que a **decisão** (positivo/negativo no 0,95) discorda
da verdade conhecida na amostra.

- **FP:** par distinto conhecido com score ≥ 0,95 (positivo errado).
- **FN:** match conhecido com score < 0,95 (negativo errado). Matches conhecidos
  que não passaram por nenhuma blocking rule não aparecem aqui — são perda de
  blocking, não de scoring.


In [ ]:
THRESHOLD_AVALIACAO = 0.95

records_fp = linker.evaluation.prediction_errors_from_labels_table(
    labels_sdf,
    threshold_match_probability=THRESHOLD_AVALIACAO,
    include_false_negatives=False,
    include_false_positives=True,
).as_record_dict(limit=20)
print('Falsos positivos (amostra):', len(records_fp))
if records_fp:
    display(linker.visualisations.waterfall_chart(records_fp, filter_nulls=False))


In [ ]:
records_fn = linker.evaluation.prediction_errors_from_labels_table(
    labels_sdf,
    threshold_match_probability=THRESHOLD_AVALIACAO,
    include_false_negatives=True,
    include_false_positives=False,
).as_record_dict(limit=20)
print('Falsos negativos (amostra):', len(records_fn))
if records_fn:
    display(linker.visualisations.waterfall_chart(records_fn, filter_nulls=False))


In [ ]:
import pandas as pd

metricas = pd.DataFrame([{
    'threshold_avaliacao': THRESHOLD_AVALIACAO,
    'n_matches_conhecidos': int((df_labels['clerical_match_score'] == 1.0).sum()),
    'n_pares_distintos': int((df_labels['clerical_match_score'] == 0.0).sum()),
}])
metricas.to_csv(METRICAS_COHORT, index=False)
print('Métricas (labels) salvas:', METRICAS_COHORT)
con.close()
